# FAIR-CARE Survey Text Response Tokens and Selection Options

This Notebook explores how tokens generated from the free-text responses compare across different groups defined by responses to selected option questions.


In [1]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

import os

import pandas as pd

from slugify import slugify

# Get the root_path for this jupyter notebook repo.
repo_path = os.path.dirname(os.path.abspath(os.getcwd()))
# Use this path to save the wordcloud outputs
wordcloud_2_path = os.path.join(
    repo_path, 'files', 'IMLS-FAIR-CARE-Survey', 'wordcloud-figs-select-group-text-questions',
)
wordcloud_ind_questions_path = os.path.join(
    repo_path, 'files', 'IMLS-FAIR-CARE-Survey', 'wordclouds-figs-select-group-text-individual-questions',
)
col_config_path = os.path.join(
    repo_path, 'files', 'IMLS-FAIR-CARE-Survey', 'imls-fair-care-survey-columns-config.csv',
)

processed_survey_path = '/home/ekansa/oc-data/fair-care-survey-processed.csv' # Keep this OUT of version control, has sensitive info
wide_survey_path = '/home/ekansa/oc-data/fair-care-survey-multi-select-expanded.csv' # Keep this OUT of version control, has sensitive info
col_summary_path = '/home/ekansa/oc-data/fair-care-survey-col-summary.csv'
token_freq_path = '/home/ekansa/oc-data/fair-care-token-freq.csv'

df_orig = pd.read_csv(processed_survey_path, low_memory=False)
df_config = pd.read_csv(col_config_path, low_memory=False)
df_token = pd.read_csv(token_freq_path, low_memory=False)
df_col_sum = pd.read_csv(col_summary_path, low_memory=False)
df_wide = pd.read_csv(wide_survey_path, low_memory=False)

print(f'FAIR+CARE survey token extract has {len(df_token.index)} rows')
print(f'FAIR+CARE survey derived multi-select -wide- has {len(df_wide.index)} rows and {len(df_wide.columns)} columns')

FAIR+CARE survey token extract has 37179 rows
FAIR+CARE survey derived multi-select -wide- has 787 rows and 271 columns


In [2]:
def get_column_total_nonblank_response_count(col, filter_index=None, df_orig=df_orig):
    """Gets the total number of nonblank reponses to a specific column"""
    if not col in df_orig.columns.tolist():
        return None
    if not filter_index:
        filter_index = ~df_orig['Response ID'].isnull()
    col_index = filter_index & ~df_orig[col].isnull()
    return len(df_orig[col_index].index)


def get_original_column_metadata(column, df_config=df_config):
    """Gets the original column name from the df_config (column configuration data)"""
    config_index = (df_config['Working_Column_Name'] == column)
    if len(df_config[config_index].index) != 1:
        # We didn't find a matching column name
        return None
    row = df_config[config_index].iloc[0]
    keys = ['column', 'orig_column_index', 'Question Number', 'Section', 'Sub-Section', 'raw_column',]
    col_metadata = {k: row[k] for k in keys if k in df_config.columns.tolist()}
    return col_metadata


def get_original_column_name(orig_column_index, df_config=df_config):
    """Gets the original column name from the df_config (column configuration data)"""
    config_index = (df_config['orig_column_index'] == orig_column_index)
    if len(df_config[config_index].index) != 1:
        # We didn't find a matching column name
        return None
    row = df_config[config_index].iloc[0]
    return row['raw_column']


def get_select_option_value_from_tf_col(tf_col):
    """Gets the select option value from a TF column"""
    col_split = tf_col.split('::')
    select_option_value = col_split[1] # The second element.
    return select_option_value


def insert_line_breaks(text, max_chars):
    """Insert line breaks for long text"""
    words = text.split()  # Split the text into words
    current_line = []
    result = []
    for word in words:
        # Check if adding the next word exceeds the max_chars limit
        if len(' '.join(current_line + [word])) > max_chars:
            # Join the current line and add it to the result list
            result.append(' '.join(current_line))
            # Start a new line with the current word
            current_line = [word]
        else:
            # Add the word to the current line
            current_line.append(word)
    # Add the last line to the result list
    if current_line:
        result.append(' '.join(current_line))
    # Join all lines with a newline character
    return '\n'.join(result)



In [3]:
def make_wordcloud_text_dict_from_df_wc(act_filter, df_wc):
    """Makes a wordcloud text dict keyed by token, with occurance counts from an index and dataframe"""
    df_wc_grp = df_wc[act_filter][['token', 'token_count']].groupby(
        ['token',], 
        as_index=False
    ).agg(
        {
            'token': 'first',
            'token_count': 'sum',
        }
    )
    df_wc_grp.sort_values(by=['token_count', 'token'], ascending=[False, True], inplace=True)
    df_wc_grp_head = df_wc_grp.head(500)
    text_dict = {}
    df_wc_grp_head = df_wc_grp.head(500)
    for _, row in df_wc_grp_head.iterrows():
        if row['token'] == 'data':
            # not very interesting
            continue
        text_dict[row['token']] = row['token_count']
    return text_dict


# Now Make WordClouds for each column, and each cluster
def create_wordcloud(text_dict, title=None, caption=None, file_suffix='', save_dir=wordcloud_2_path):
    plt.rcParams["figure.figsize"] = (6, 6)
    wc = WordCloud(background_color="white", max_words=500, width=500, height=300)
    wc.generate_from_frequencies(text_dict)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    if title:
        plt.title(title)
    if caption:
        plt.figtext(0.5, 0.01, caption, wrap=True, horizontalalignment='center', fontsize=9)
    slug_file = slugify(title)
    plt.autoscale()
    filename = f'{slug_file}{file_suffix}.png'
    f_path = os.path.join(save_dir, filename)
    plt.savefig(f_path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f'Saved figure: {filename}')


In [4]:
# Columns used to identify responses from CRM settings
crm_cols = [
    'Work: Work setting::CRM: Cultural Resources Consulting Firm::TF',
    'Work: Work setting::CRM: Tribally-Owned or similar Consulting Firm::TF',
    'Work: Work setting::CRM: Museum or University-based Consulting Organization::TF',
    'Work: Work setting::Construction Management Firm::TF',
    'Work: Work setting::CRM: Environmental or Engineering Consulting Firm::TF',
]

wide_cols = [
    'Response ID', 
    'Demo: Age Range',	
    'Demo: Gender', 
    'Work: Work setting::Academic: University or College::TF',
    'Region Focus: Work/research Region::North America (specify region)::TF',
    'Response Type: Individual or Org ',
] + crm_cols

# Merge certain columns from the df_wide into the df_token dataset so we can
# filter by criteria from the df_wide.
df_wc = pd.merge(df_token, df_wide[wide_cols], how='left', on=['Response ID'])


In [5]:
# Make an index to select responses that indicate work in ANY CRM setting (lots of 'or' selections)
any_crm_index = (df_wc[crm_cols[0]] == True)
for crm_col in crm_cols[1:]:
    # Add an "or" option for the next CRM column
    any_crm_index |= (df_wc[crm_col] == True)



filtered_plot_configs = [
    # (filter_index, title_suffix, file_suffix,),
    ((~df_wc['Response ID'].isnull()), '\nAll Reponses (Unfiltered)', '-all',),
    ((df_wc['Work: Work setting::Academic: University or College::TF'] == True), '\nAcademic (Univ)', '-academic',),
    ((df_wc['Work: Work setting::Academic: University or College::TF'] == False), '\nNot Academic (Univ)', '-not-academic',),
    (any_crm_index, '\nCRM', '-crm',),
    ((df_wc['Region Focus: Work/research Region::North America (specify region)::TF'] == True), '\nN. America Focus', '-n-america',),
    ((df_wc['Region Focus: Work/research Region::North America (specify region)::TF'] == False), '\nOutside N. America Focus', '-out-n-america',),
    ((df_wc['Response Type: Individual or Org '] == 'Individual'), '\nResponding as Individual', '-resp-ind',),
    ((df_wc['Response Type: Individual or Org '].str.contains('Organization')), '\nResponding as Organization', '-resp-org',),
]

# Make groups of different sections, sub-sections that we'd like to use for wordclouds
use_index = df_wc['orig_column_index'] >= 38
df_sec_grp = df_wc[use_index][['Section', 'Sub-Section']].groupby(['Section', 'Sub-Section'], as_index=False).first()


In [6]:
for filter_index, title_suffix, file_suffix in filtered_plot_configs:
    for _, gr_row in df_sec_grp.iterrows():
        section = gr_row['Section']
        sub_section = gr_row['Sub-Section']
        act_filter = filter_index & (df_wc['Section'] == section) & (df_wc['Sub-Section'] == sub_section)
        if len(df_wc[act_filter].index) < 1:
            # No need to make a wordcloud from empty data!
            continue
        count_respondants = df_wc[act_filter]['Response ID'].unique().size
        count_questions = df_wc[act_filter]['column'].unique().size
        sum_act_tokens = df_wc[act_filter]['token_count'].sum()
        title = f'{section} -- {sub_section} Words from {title_suffix}'
        caption = f'(From n = {count_respondants} individual respondants; {count_questions} freetext questions)'
        text_dict = make_wordcloud_text_dict_from_df_wc(act_filter, df_wc)
        create_wordcloud(text_dict, title, caption, file_suffix=file_suffix)
        all_act_filter = (~df_wc['Response ID'].isnull()) & (df_wc['Section'] == section) & (df_wc['Sub-Section'] == sub_section)
        sum_all_tokens = df_wc[all_act_filter]['token_count'].sum()
        all_text_dict = make_wordcloud_text_dict_from_df_wc(all_act_filter, df_wc)
        filtered_text_dict = {}
        for token, count in text_dict.items():
            act_token_rate = count / sum_act_tokens
            all_token_rate = all_text_dict.get(token, 0) / sum_all_tokens
            # the expected count for this token is derived from the relative
            # frequency of this token from the unfiltered set of responses
            expected_count = sum_act_tokens * all_token_rate
            count_dif_expected = count - expected_count
            if count_dif_expected <= 1:
                # The difference from the expected count is less than one,
                # so don't display this token in a wordcloud
                continue
            count_dif_expected = int(round(count_dif_expected, 0))
            filtered_text_dict[token] = count_dif_expected
        if not filtered_text_dict:
            # None of the tokens fit our criteria for making a filtered, wordcloud.
            continue
        title += ' [Reweighted for Distinctive]'
        filtered_file_suffix = file_suffix + '-weighted'
        create_wordcloud(filtered_text_dict, title, caption, file_suffix=filtered_file_suffix)
    

Saved figure: care-authority-to-control-words-from-all-reponses-unfiltered-all.png
Saved figure: care-collective-benefit-words-from-all-reponses-unfiltered-all.png
Saved figure: care-ethics-words-from-all-reponses-unfiltered-all.png
Saved figure: care-general-words-from-all-reponses-unfiltered-all.png
Saved figure: care-responsibility-words-from-all-reponses-unfiltered-all.png
Saved figure: demographics-data-role-words-from-all-reponses-unfiltered-all.png
Saved figure: fair-accessible-words-from-all-reponses-unfiltered-all.png
Saved figure: fair-findable-words-from-all-reponses-unfiltered-all.png
Saved figure: fair-interoperable-words-from-all-reponses-unfiltered-all.png
Saved figure: fair-reusable-words-from-all-reponses-unfiltered-all.png
Saved figure: general-general-words-from-all-reponses-unfiltered-all.png
Saved figure: care-authority-to-control-words-from-academic-univ-academic.png
Saved figure: care-authority-to-control-words-from-academic-univ-reweighted-for-distinctive-academ

In [7]:
for filter_index, title_suffix, file_suffix in filtered_plot_configs:
    # iterate through each of the text response columns.
    for col in df_wc[use_index]['column'].unique().tolist():
        col_index = (df_wc['column'] == col)
        act_filter = filter_index & col_index
        if len(df_wc[act_filter].index) < 1:
            # No need to make a wordcloud from empty data!
            continue
        count_respondants = df_wc[act_filter]['Response ID'].unique().size
        sum_act_tokens = df_wc[act_filter]['token_count'].sum()
        section = df_wc[act_filter]['Section'].iloc[0]
        sub_section = df_wc[act_filter]['Sub-Section'].iloc[0]
        question = df_wc[act_filter]['Question Number'].iloc[0]
        orig_column_index = df_wc[act_filter]['orig_column_index'].iloc[0]
        orig_col_name  = get_original_column_name(orig_column_index)
        question = int(question)
        title = f'{section} -- {sub_section} ({question}) {col}{title_suffix}'
        caption = f'[Question: {question}] "{orig_col_name}"\n(n = {count_respondants} responses)'
        text_dict = make_wordcloud_text_dict_from_df_wc(act_filter=act_filter, df_wc=df_wc)
        if file_suffix == '-all':
            # Only make the non-distinct wordcloud for un-segmented responses to the question
            # otherwise, we'll end up with too many plots.
            create_wordcloud(text_dict, title, caption, save_dir=wordcloud_ind_questions_path)
        # Now make a filter for all the tokens used in a given question
        all_act_filter = col_index
        sum_all_tokens = df_wc[all_act_filter]['token_count'].sum()
        all_text_dict = make_wordcloud_text_dict_from_df_wc(all_act_filter, df_wc)
        filtered_text_dict = {}
        for token, count in text_dict.items():
            act_token_rate = count / sum_act_tokens
            all_token_rate = all_text_dict.get(token, 0) / sum_all_tokens
            # the expected count for this token is derived from the relative
            # frequency of this token from the unfiltered set of responses
            expected_count = sum_act_tokens * all_token_rate
            count_dif_expected = count - expected_count
            if count_dif_expected <= 1:
                # The difference from the expected count is less than one,
                # so don't display this token in a wordcloud
                continue
            count_dif_expected = int(round(count_dif_expected, 0))
            filtered_text_dict[token] = count_dif_expected
        if not filtered_text_dict:
            # None of the tokens fit our criteria for making a filtered, wordcloud.
            continue
        title += ' [Reweighted for Distinctive]'
        filtered_file_suffix = file_suffix + '-weighted'
        create_wordcloud(filtered_text_dict, title, caption, file_suffix=filtered_file_suffix, save_dir=wordcloud_ind_questions_path)
        
        

Saved figure: care-ethics-40-application-of-ethical-frameworks-text-all-reponses-unfiltered.png
Saved figure: care-general-21-care-define-indigenous-data-text-all-reponses-unfiltered.png
Saved figure: care-authority-to-control-34-collaboration-processes-collaboration-processes-exist-yes-no-text-all-reponses-unfiltered.png
Saved figure: care-authority-to-control-29-control-policies-polices-for-indigenous-use-refusal-reclaim-data-no-text-all-reponses-unfiltered.png
Saved figure: care-ethics-41-data-sensitivities-text-all-reponses-unfiltered.png
Saved figure: care-collective-benefit-25-disclosure-process-indigenous-and-or-descendant-disclosure-examples-text-all-reponses-unfiltered.png
Saved figure: care-responsibility-38-do-you-attribute-yes-no-text-all-reponses-unfiltered.png
Saved figure: fair-findable-4-findable-yes-text-all-reponses-unfiltered.png
Saved figure: care-ethics-42-inclusive-interpretation-and-presentation-yes-text-all-reponses-unfiltered.png
Saved figure: care-authority-to